1. Setup and Dependencies

In [2]:

# Install required packages (run only once)
import subprocess
import sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '--quiet'])

required_packages = [
    'pandas', 'mysql-connector-python', 'numpy', 'scikit-learn',
    'networkx', 'matplotlib', 'torch', 'pyvis'
]

for pkg in required_packages:
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        install(pkg)

# Imports
import pandas as pd
import mysql.connector
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as F
import json
import os
from datetime import datetime

# Set display options
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)



2. Database Connection

In [3]:
# Connect to the database
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Alok@12&kumar#@",
    database="railtel"
)
print("Connected")

Connected


3. Load and Explore Tables

In [4]:
# Get list of tables
tables_df = pd.read_sql("SHOW TABLES", conn)
table_names = tables_df.iloc[:, 0].tolist()
print(f"Total tables: {len(table_names)}")

Total tables: 143


C:\Users\HP\AppData\Local\Temp\ipykernel_2812\2154117218.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  tables_df = pd.read_sql("SHOW TABLES", conn)


In [5]:
required_tables = [
    "alarm", "network_element", "bgp_link", "lldp_link", "isis_link", "ospf_link",
    "ne_hardware_details", "interface_status_detail", "alarm_library", "ne_location",
    "network_service", "ckt_id_info", "lsp", "lsp_hop", "network_service_history"
]

# Filter only existing tables
filtered_tables = [t for t in table_names if t in required_tables]
print(f"Selected tables: {filtered_tables}")
print(f"Count: {len(filtered_tables)}")

Selected tables: ['alarm', 'alarm_library', 'bgp_link', 'ckt_id_info', 'interface_status_detail', 'isis_link', 'lldp_link', 'lsp', 'lsp_hop', 'ne_hardware_details', 'ne_location', 'network_element', 'network_service', 'network_service_history', 'ospf_link']
Count: 15


In [6]:
data = {}
for table in filtered_tables:
    print(f"Loading {table}...")
    try:
        data[table] = pd.read_sql(f"SELECT * FROM {table}", conn)
        print(f"  {table}: {data[table].shape}")
    except Exception as e:
        print(f"  ERROR loading {table}: {e}")
        data[table] = pd.DataFrame()
print("Done.")

Loading alarm...


C:\Users\HP\AppData\Local\Temp\ipykernel_2812\269429969.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data[table] = pd.read_sql(f"SELECT * FROM {table}", conn)


  alarm: (663510, 72)
Loading alarm_library...
  alarm_library: (8814, 44)
Loading bgp_link...
  bgp_link: (2077, 20)
Loading ckt_id_info...
  ckt_id_info: (33246, 5)
Loading interface_status_detail...
  interface_status_detail: (0, 14)
Loading isis_link...
  isis_link: (294, 45)
Loading lldp_link...
  lldp_link: (1581, 19)
Loading lsp...
  lsp: (101, 9)
Loading lsp_hop...
  lsp_hop: (398, 9)
Loading ne_hardware_details...


C:\Users\HP\AppData\Local\Temp\ipykernel_2812\269429969.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data[table] = pd.read_sql(f"SELECT * FROM {table}", conn)


  ne_hardware_details: (32349, 23)
Loading ne_location...
  ne_location: (6749, 52)
Loading network_element...
  network_element: (59667, 70)
Loading network_service...
  network_service: (12660, 57)
Loading network_service_history...
  network_service_history: (195360, 25)
Loading ospf_link...
  ospf_link: (1138, 23)
Done.


4. Data Cleaning

create a generic cleaning function and apply it to each table

In [7]:
def clean_table(df, timestamp_cols=None, drop_nulls_on=None):
    """
    Clean a DataFrame: drop duplicates, convert timestamps, fill NaNs.
    """
    if df.empty:
        return df
    df = df.copy()
    # Remove duplicate rows
    df = df.drop_duplicates()
    # Convert timestamp columns
    for col in timestamp_cols or []:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')
    # Fill missing values: strings -> 'UNKNOWN', numbers -> 0
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].fillna('UNKNOWN')
        else:
            df[col] = df[col].fillna(0)
    # Drop rows with nulls in essential columns
    if drop_nulls_on:
        df = df.dropna(subset=[c for c in drop_nulls_on if c in df.columns])
    return df

# Clean each table
for table, df in data.items():
    if table == 'alarm':
        data[table] = clean_table(df,
            timestamp_cols=['OPEN_TIME', 'CLOSURE_TIME'],
            drop_nulls_on=['ALARM_ID_PK', 'OPEN_TIME', 'ENTITY_ID'])
    elif table == 'network_element':
        data[table] = clean_table(df, drop_nulls_on=['NE_ID'])
    else:
        data[table] = clean_table(df)

print("All tables cleaned.")

All tables cleaned.


5. Column Selection

In [8]:
# Alarm columns
alarm_cols = [
    'ALARM_ID_PK', 'OPEN_TIME', 'CLOSURE_TIME', 'SEVERITY', 'ACTUAL_SEVERITY',
    'ALARM_STATUS', 'ALARM_CODE', 'ALARM_NAME', 'EVENT_TYPE', 'ENTITY_NAME',
    'ENTITY_ID', 'PARENT_ENTITY_ID', 'ENTITY_TYPE', 'VENDOR', 'TECHNOLOGY',
    'DOMAIN', 'PROBABLE_CAUSE', 'CORRELATION_TYPE', 'CORRELATION_FLAG',
    'INCIDENT_ID', 'PARTICIPATED_IN_INCIDENT', 'INCIDENT_STATUS',
    'GEOGRAPHY_L1_NAME', 'GEOGRAPHY_L2_NAME', 'GEOGRAPHY_L3_NAME',
    'GEOGRAPHY_L4_NAME', 'CKT_ID', 'SERVICE_AFFECTED', 'DESCRIPTION'
]
alarm_cols = [c for c in alarm_cols if c in data['alarm'].columns]
data['alarm'] = data['alarm'][alarm_cols]

# Network element columns
ne_cols = [
    'ID', 'NE_ID', 'NE_NAME', 'NE_TYPE', 'VENDOR', 'TECHNOLOGY',
    'IP_ADDRESS', 'DOMAIN', 'STATUS', 'LOCATION_ID'
]
ne_cols = [c for c in ne_cols if c in data['network_element'].columns]
data['network_element'] = data['network_element'][ne_cols]

# Topology tables
topo_cols = ['SOURCE_NE_ID', 'DESTINATION_NE_ID', 'STATUS', 'CREATED_TIME', 'MODIFIED_TIME', 'CKT_ID']
for tname in ['bgp_link', 'isis_link', 'ospf_link']:
    cols = [c for c in topo_cols if c in data[tname].columns]
    data[tname] = data[tname][cols]

# lldp_link uses interface IDs
lldp_cols = ['SOURCE_INTERFACE_NE_ID', 'DESTINATION_INTERFACE_NE_ID', 'CREATED_TIME', 'MODIFIED_TIME', 'CKT_ID']
lldp_cols = [c for c in lldp_cols if c in data['lldp_link'].columns]
data['lldp_link'] = data['lldp_link'][lldp_cols]

# Other tables (select important columns)
al_lib_cols = [
    'ALARM_LIBRARY_ID_PK', 'ALARM_IDENTIFIER', 'ALARM_NAME', 'ALARM_HIERARCHY',
    'CORRELATION_ENABLE', 'SERVICE_AFFECTING', 'CONTRIBUTOR_CATEGORY',
    'PROBABLE_CAUSE', 'ALARM_LAYER', 'ALARM_GROUP', 'DEFAULT_SEVERITY',
    'DESCRIPTION', 'SUGGESTION'
]
al_lib_cols = [c for c in al_lib_cols if c in data['alarm_library'].columns]
data['alarm_library'] = data['alarm_library'][al_lib_cols]

loc_cols = ['ID', 'NE_ID_FK', 'LOCATION_NAME', 'LATITUDE', 'LONGITUDE']
loc_cols = [c for c in loc_cols if c in data['ne_location'].columns]
data['ne_location'] = data['ne_location'][loc_cols]

iface_cols = ['ID', 'NE_ID_FK', 'INTERFACE_NAME', 'ADMIN_STATE', 'OPERATIONAL_STATE', 'BANDWIDTH', 'IP_ADDRESS']
iface_cols = [c for c in iface_cols if c in data['interface_status_detail'].columns]
data['interface_status_detail'] = data['interface_status_detail'][iface_cols]

hw_cols = ['ID', 'NE_ID_FK', 'MODEL', 'VENDOR', 'SERIAL_NUMBER', 'HARDWARE_STATUS']
hw_cols = [c for c in hw_cols if c in data['ne_hardware_details'].columns]
data['ne_hardware_details'] = data['ne_hardware_details'][hw_cols]

ns_cols = [
    'ID', 'NETWORK_SERVICE_NAME', 'STATUS', 'OPERATIONAL_STATUS', 'ADMIN_STATUS',
    'CKT_ID', 'SOURCE_NE_ID', 'NETWORK_ELEMENT_ID_FK', 'SERVICE_TYPE',
    'TECHNOLOGY', 'CUSTOMER_NAME'
]
ns_cols = [c for c in ns_cols if c in data['network_service'].columns]
data['network_service'] = data['network_service'][ns_cols]

# Additional tables for labeling
ckt_cols = ['ID', 'NETWORK_ELEMENT_ID_FK', 'CKT_ID', 'IS_DELETED', 'CREATED_TIME']
if 'ckt_id_info' in data and not data['ckt_id_info'].empty:
    ckt_cols = [c for c in ckt_cols if c in data['ckt_id_info'].columns]
    data['ckt_id_info'] = data['ckt_id_info'][ckt_cols]

lsp_cols = ['ID', 'LSP_NAME', 'SOURCE_IP', 'SOURCE_NE_ID', 'DESTINATION_IP', 'DESTINATION_NE_ID']
if 'lsp' in data and not data['lsp'].empty:
    lsp_cols = [c for c in lsp_cols if c in data['lsp'].columns]
    data['lsp'] = data['lsp'][lsp_cols]

lsp_hop_cols = ['ID', 'LSP_ID', 'HOP_SEQUENCE', 'IF_IP', 'ROUTER_NE_ID', 'INTERFACE_NE_ID']
if 'lsp_hop' in data and not data['lsp_hop'].empty:
    lsp_hop_cols = [c for c in lsp_hop_cols if c in data['lsp_hop'].columns]
    data['lsp_hop'] = data['lsp_hop'][lsp_hop_cols]

ns_hist_cols = ['ID', 'NS_ID', 'LAST_DISCOVERY_TIME', 'STATUS', 'CKT_ID', 'MODIFIED_TIME']
if 'network_service_history' in data and not data['network_service_history'].empty:
    ns_hist_cols = [c for c in ns_hist_cols if c in data['network_service_history'].columns]
    data['network_service_history'] = data['network_service_history'][ns_hist_cols]

6. Key Join: Linking Alarms to Network Elements

Alarm ENTITY_ID may be an IP address or a numeric NE_ID. We'll create a unified mapping.

In [9]:
ne_clean = data['network_element']
alarm_clean = data['alarm']

# Create mapping from NE_ID (string) and IP_ADDRESS to node index
# Also include the numeric ID (if available) to match foreign keys in topology tables
ne_id_to_idx = {}
for idx, row in ne_clean.iterrows():
    # Primary key: NE_ID (string)
    ne_id_str = str(row['NE_ID'])
    ne_id_to_idx[ne_id_str] = idx
    # Also map IP_ADDRESS if present
    if 'IP_ADDRESS' in row and row['IP_ADDRESS'] not in ('UNKNOWN', '0', 'nan'):
        ne_id_to_idx[str(row['IP_ADDRESS'])] = idx
    # Map numeric ID (if present) as string to cover integer FKs
    if 'ID' in row and row['ID'] not in (0, '0', 'UNKNOWN'):
        ne_id_to_idx[str(row['ID'])] = idx

# Filter alarms that map to a known network element
alarm_clean['ENTITY_ID_STR'] = alarm_clean['ENTITY_ID'].astype(str)
mask = alarm_clean['ENTITY_ID_STR'].isin(ne_id_to_idx)
alarm_clean = alarm_clean[mask].copy()
print(f"Alarms after mapping: {len(alarm_clean)} rows")

Alarms after mapping: 325170 rows


7. Graph Construction

a graph where nodes are network elements (by NE_ID) and edges come from topology tables.

In [10]:
# Node features: encode categorical columns, add hardware/interface flags
# Categorical columns to encode
cat_cols = ['NE_TYPE', 'VENDOR', 'TECHNOLOGY', 'DOMAIN', 'STATUS']
for col in cat_cols:
    if col in ne_clean.columns:
        le = LabelEncoder()
        ne_clean[col + '_ENC'] = le.fit_transform(ne_clean[col].astype(str))
    else:
        ne_clean[col + '_ENC'] = 0

# Hardware failure flag
if 'ne_hardware_details' in data and not data['ne_hardware_details'].empty:
    hw_df = data['ne_hardware_details']
    if 'NE_ID_FK' in hw_df.columns and 'HARDWARE_STATUS' in hw_df.columns:
        failed_hw = set(hw_df[hw_df['HARDWARE_STATUS'] != 'NORMAL']['NE_ID_FK'].dropna())
        # Map to node index via NE_ID (string) or ID (int)
        ne_clean['HW_ISSUE'] = ne_clean.apply(
            lambda row: 1 if (str(row['NE_ID']) in failed_hw or
                              (row.get('ID') and str(row['ID']) in failed_hw))
            else 0, axis=1)
    else:
        ne_clean['HW_ISSUE'] = 0
else:
    ne_clean['HW_ISSUE'] = 0

# Interface down count
if 'interface_status_detail' in data and not data['interface_status_detail'].empty:
    iface_df = data['interface_status_detail']
    if 'NE_ID_FK' in iface_df.columns and 'OPERATIONAL_STATE' in iface_df.columns:
        down_counts = iface_df[iface_df['OPERATIONAL_STATE'] == 'DOWN'].groupby('NE_ID_FK').size()
        ne_clean['IF_DOWN_COUNT'] = ne_clean.apply(
            lambda row: down_counts.get(str(row['NE_ID']), 0), axis=1)
    else:
        ne_clean['IF_DOWN_COUNT'] = 0
else:
    ne_clean['IF_DOWN_COUNT'] = 0

# Assemble feature matrix
enc_cols = [c + '_ENC' for c in cat_cols if c + '_ENC' in ne_clean.columns]
feat_cols = enc_cols + ['HW_ISSUE', 'IF_DOWN_COUNT']
node_feats = ne_clean[feat_cols].values.astype(np.float32)
# Add two placeholder columns for alarm activity (will be updated per incident)
node_feats = np.hstack([node_feats, np.zeros((node_feats.shape[0], 2))])
print(f"Node feature matrix shape: {node_feats.shape}")

# Build edges from topology tables
def add_edges_from_table(df, src_col, dst_col, edge_list):
    if df.empty:
        return
    s = df[src_col].astype(str).map(ne_id_to_idx)
    d = df[dst_col].astype(str).map(ne_id_to_idx)
    valid = s.notna() & d.notna() & (s != d)
    for si, di in zip(s[valid], d[valid]):
        edge_list.append((int(si), int(di)))

edges = []
# BGP, ISIS, OSPF use SOURCE_NE_ID / DESTINATION_NE_ID
for tname in ['bgp_link', 'isis_link', 'ospf_link']:
    df = data.get(tname, pd.DataFrame())
    add_edges_from_table(df, 'SOURCE_NE_ID', 'DESTINATION_NE_ID', edges)
# LLDP uses interface IDs; we assume those IDs can be mapped via NE_ID as well
df = data.get('lldp_link', pd.DataFrame())
add_edges_from_table(df, 'SOURCE_INTERFACE_NE_ID', 'DESTINATION_INTERFACE_NE_ID', edges)

# Convert to undirected (add both directions)
edge_index = []
for u, v in edges:
    edge_index.append([u, v])
    edge_index.append([v, u])
edge_index = torch.tensor(edge_index, dtype=torch.long).t() if edge_index else torch.zeros((2, 0), dtype=torch.long)

print(f"Graph: {node_feats.shape[0]} nodes, {edge_index.shape[1]//2} undirected edges")

Node feature matrix shape: (59667, 9)
Graph: 59667 nodes, 294 undirected edges


8. Label Engineering Strategies

8.1. Method 1: Alarm Library Hierarchy

In [11]:
al_lib = data.get('alarm_library', pd.DataFrame())
labels_m1 = pd.DataFrame()
if not al_lib.empty and 'ALARM_HIERARCHY' in al_lib.columns:
    # Identify root-type alarm codes
    root_mask = al_lib['ALARM_HIERARCHY'].astype(str).str.upper().str.contains('ROOT|PRIMARY', na=False)
    if 'CORRELATION_ENABLE' in al_lib.columns:
        root_mask = root_mask & (al_lib['CORRELATION_ENABLE'].astype(str).str.upper() != 'FALSE')
    root_codes = set(al_lib[root_mask]['ALARM_IDENTIFIER'].dropna())
    if root_codes:
        root_alarms = alarm_clean[alarm_clean['ALARM_CODE'].isin(root_codes) &
                                  alarm_clean['INCIDENT_ID'].notna() &
                                  (alarm_clean['INCIDENT_ID'] != 'UNKNOWN')]
        if not root_alarms.empty:
            # Take earliest alarm per incident
            root_alarms = root_alarms.sort_values('OPEN_TIME')
            first_per_inc = root_alarms.drop_duplicates(subset='INCIDENT_ID', keep='first')
            first_per_inc['ROOT_NE_ID'] = first_per_inc['ENTITY_ID_STR']
            # Map to NE name (optional)
            ne_map = ne_clean.set_index('NE_ID')['NE_NAME'].to_dict()
            first_per_inc['ROOT_NE_NAME'] = first_per_inc['ROOT_NE_ID'].map(ne_map)
            labels_m1 = first_per_inc[['INCIDENT_ID', 'ROOT_NE_ID', 'ROOT_NE_NAME']].copy()
            labels_m1['LABEL_SOURCE'] = 'M1_alarm_library'

8.2. Method 2: Incident ID and Correlation Type

In [12]:
labels_m2 = pd.DataFrame()
if 'INCIDENT_ID' in alarm_clean.columns and 'CORRELATION_TYPE' in alarm_clean.columns:
    # Try to find ROOT correlation type
    root_corr = alarm_clean[alarm_clean['CORRELATION_TYPE'].astype(str).str.upper().str.contains('ROOT|CAUSE', na=False)]
    if root_corr.empty:
        # Fallback: earliest alarm per incident
        root_corr = alarm_clean.sort_values('OPEN_TIME').drop_duplicates(subset='INCIDENT_ID', keep='first')
    else:
        root_corr = root_corr.sort_values('OPEN_TIME').drop_duplicates(subset='INCIDENT_ID', keep='first')
    if not root_corr.empty:
        root_corr['ROOT_NE_ID'] = root_corr['ENTITY_ID_STR']
        ne_map = ne_clean.set_index('NE_ID')['NE_NAME'].to_dict()
        root_corr['ROOT_NE_NAME'] = root_corr['ROOT_NE_ID'].map(ne_map)
        labels_m2 = root_corr[['INCIDENT_ID', 'ROOT_NE_ID', 'ROOT_NE_NAME']].copy()
        labels_m2['LABEL_SOURCE'] = 'M2_incident_id'

8.3. Method 3: CKT_ID Bridge

In [13]:
labels_m3 = pd.DataFrame()
ckt_df = data.get('ckt_id_info', pd.DataFrame())
if not ckt_df.empty and 'CKT_ID' in ckt_df.columns and 'NETWORK_ELEMENT_ID_FK' in ckt_df.columns:
    # Convert CKT_ID to string for merge
    alarm_clean['CKT_ID_STR'] = alarm_clean['CKT_ID'].astype(str)
    ckt_df['CKT_ID_STR'] = ckt_df['CKT_ID'].astype(str)
    # Merge
    merged = alarm_clean.merge(ckt_df, on='CKT_ID_STR', how='inner')
    if not merged.empty:
        # Convert NETWORK_ELEMENT_ID_FK to string for mapping
        merged['ROOT_NE_ID'] = merged['NETWORK_ELEMENT_ID_FK'].astype(str)
        # Map to NE_NAME
        ne_map = ne_clean.set_index('NE_ID')['NE_NAME'].to_dict()
        merged['ROOT_NE_NAME'] = merged['ROOT_NE_ID'].map(ne_map)
        # Keep earliest per incident
        merged = merged.sort_values('OPEN_TIME').drop_duplicates(subset='INCIDENT_ID', keep='first')
        labels_m3 = merged[['INCIDENT_ID', 'ROOT_NE_ID', 'ROOT_NE_NAME']].copy()
        labels_m3['LABEL_SOURCE'] = 'M3_ckt_bridge'

8.4. Method 4: Service Down Events

In [14]:
labels_m4 = pd.DataFrame()
ns_hist = data.get('network_service_history', pd.DataFrame())
if not ns_hist.empty and 'STATUS' in ns_hist.columns and 'MODIFIED_TIME' in ns_hist.columns:
    ns_hist['MODIFIED_TIME'] = pd.to_datetime(ns_hist['MODIFIED_TIME'], errors='coerce')
    down_events = ns_hist[ns_hist['STATUS'].astype(str).str.upper().str.contains('DOWN|INACTIVE', na=False)]
    down_events = down_events.dropna(subset=['MODIFIED_TIME', 'CKT_ID'])
    # Process each down event
    rows = []
    for _, ev in down_events.iterrows():
        t_down = ev['MODIFIED_TIME']
        ckt = str(ev['CKT_ID'])
        window_start = t_down - pd.Timedelta(minutes=5)
        window_alarms = alarm_clean[(alarm_clean['OPEN_TIME'] >= window_start) &
                                    (alarm_clean['OPEN_TIME'] <= t_down) &
                                    (alarm_clean['CKT_ID_STR'] == ckt)]
        if window_alarms.empty:
            continue
        first_alarm = window_alarms.sort_values('OPEN_TIME').iloc[0]
        rows.append({
            'INCIDENT_ID': first_alarm['INCIDENT_ID'],
            'ROOT_NE_ID': first_alarm['ENTITY_ID_STR'],
            'ROOT_NE_NAME': ne_map.get(first_alarm['ENTITY_ID_STR'], 'UNKNOWN')
        })
    if rows:
        labels_m4 = pd.DataFrame(rows).drop_duplicates(subset='INCIDENT_ID')
        labels_m4['LABEL_SOURCE'] = 'M4_service_history'

8.5. Method 5: LSP Hop Path

In [15]:
labels_m5 = pd.DataFrame()
lsp_df = data.get('lsp', pd.DataFrame())
lsp_hop_df = data.get('lsp_hop', pd.DataFrame())
if not lsp_df.empty and not lsp_hop_df.empty and 'ROUTER_NE_ID' in lsp_hop_df.columns:
    # Join alarms with hops
    alarm_ne = alarm_clean[['ALARM_ID_PK', 'OPEN_TIME', 'INCIDENT_ID', 'ENTITY_ID_STR']].dropna(subset=['INCIDENT_ID'])
    # Convert ROUTER_NE_ID to string for merging
    lsp_hop_df['ROUTER_NE_ID_STR'] = lsp_hop_df['ROUTER_NE_ID'].astype(str)
    merged = lsp_hop_df.merge(alarm_ne, left_on='ROUTER_NE_ID_STR', right_on='ENTITY_ID_STR', how='inner')
    if not merged.empty and 'HOP_SEQUENCE' in merged.columns:
        merged = merged.sort_values(['INCIDENT_ID', 'HOP_SEQUENCE', 'OPEN_TIME'])
        first_hop = merged.drop_duplicates(subset='INCIDENT_ID', keep='first')
        first_hop['ROOT_NE_ID'] = first_hop['ROUTER_NE_ID_STR']
        ne_map = ne_clean.set_index('NE_ID')['NE_NAME'].to_dict()
        first_hop['ROOT_NE_NAME'] = first_hop['ROOT_NE_ID'].map(ne_map)
        labels_m5 = first_hop[['INCIDENT_ID', 'ROOT_NE_ID', 'ROOT_NE_NAME']].copy()
        labels_m5['LABEL_SOURCE'] = 'M5_lsp_hop'

8.6. Combine All Labels

In [16]:
all_labels = []
for df in [labels_m1, labels_m2, labels_m3, labels_m4, labels_m5]:
    if not df.empty:
        all_labels.append(df)

if all_labels:
    combined = pd.concat(all_labels, ignore_index=True)
    # Priority order (lower number = higher priority)
    priority_map = {'M1_alarm_library': 1, 'M2_incident_id': 2, 'M3_ckt_bridge': 3,
                    'M4_service_history': 4, 'M5_lsp_hop': 5}
    combined['PRIORITY'] = combined['LABEL_SOURCE'].map(priority_map)
    combined = combined.sort_values(['INCIDENT_ID', 'PRIORITY'])
    labels_df = combined.drop_duplicates(subset='INCIDENT_ID', keep='first').reset_index(drop=True)
    # Multi-method agreement flag
    agreement = combined.groupby('INCIDENT_ID')['ROOT_NE_ID'].nunique()
    labels_df['MULTI_METHOD_AGREE'] = labels_df['INCIDENT_ID'].map(lambda x: agreement.get(x, 1) == 1)
    # Save
    os.makedirs('data_pipeline', exist_ok=True)
    labels_df.to_csv('data_pipeline/labels.csv', index=False)
    print(f"Total labeled incidents: {len(labels_df)}")
    print(f"Multi-method agreement: {labels_df['MULTI_METHOD_AGREE'].sum()} ({labels_df['MULTI_METHOD_AGREE'].mean():.1%})")
else:
    labels_df = pd.DataFrame()
    print("No labels generated.")

Total labeled incidents: 343
Multi-method agreement: 326 (95.0%)


9. Temporal Dataset Creation

create sequences of alarms per incident and pair them with the root cause label.

In [ ]:
# Encode NE names to node indices
ne_encoder = LabelEncoder()
ne_encoder.fit(ne_clean['NE_NAME'].astype(str).values)
print(f"Number of nodes: {len(ne_encoder.classes_)}")

severity_map = {'CRITICAL': 4, 'MAJOR': 3, 'MINOR': 2, 'WARNING': 1, 'INFO': 0}

temporal_dataset = []
skipped = 0

for inc_id in labels_df['INCIDENT_ID'].unique():
    inc_alarms = alarm_clean[alarm_clean['INCIDENT_ID'] == inc_id].copy()
    if len(inc_alarms) < 2:
        skipped += 1
        continue
    inc_alarms = inc_alarms.sort_values('OPEN_TIME').reset_index(drop=True)
    t0 = inc_alarms['OPEN_TIME'].iloc[0]

    events = []
    for _, row in inc_alarms.iterrows():
        ne_name = str(row.get('ENTITY_NAME', ''))
        if ne_name not in ne_encoder.classes_:
            continue
        node_idx = ne_encoder.transform([ne_name])[0]
        rel_time = (row['OPEN_TIME'] - t0).total_seconds()
        sev = severity_map.get(str(row.get('SEVERITY', 'WARNING')), 1)
        events.append((node_idx, rel_time, sev))

    if len(events) < 2:
        skipped += 1
        continue

    # Get label
    label_info = labels_df[labels_df['INCIDENT_ID'] == inc_id].iloc[0]
    root_ne_name = str(label_info['ROOT_NE_NAME'])
    if root_ne_name not in ne_encoder.classes_:
        skipped += 1
        continue
    label_idx = ne_encoder.transform([root_ne_name])[0]

    temporal_dataset.append((
        torch.tensor(node_feats, dtype=torch.float32),  # node features
        edge_index,                                     # graph edges
        events,                                         # alarm events
        label_idx                                       # root cause node index
    ))

print(f"Dataset samples: {len(temporal_dataset)} | Skipped: {skipped}")

# Split
if len(temporal_dataset) >= 6:
    idx = list(range(len(temporal_dataset)))
    train_idx, temp_idx = train_test_split(idx, test_size=0.3, random_state=42)
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)
else:
    train_idx = val_idx = test_idx = idx

print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

Number of nodes: 59667


10. TGAT Model

Temporal Graph Attention Network that uses node features and alarm events.

In [ ]:
class TimeEncoding(nn.Module):
    def __init__(self, dim=64):
        super().__init__()
        self.omega = nn.Parameter(torch.randn(dim // 2))
        self.dim = dim
    def forward(self, t):
        if not torch.is_tensor(t):
            t = torch.tensor(t, dtype=torch.float32)
        t = t.float()
        angles = t.unsqueeze(-1) * self.omega.unsqueeze(0)
        return torch.cat([torch.cos(angles), torch.sin(angles)], dim=-1)

class TGAT(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, heads=4, time_dim=64):
        super().__init__()
        self.time_enc = TimeEncoding(time_dim)
        self.in_proj = nn.Linear(in_dim + time_dim, hidden_dim)
        self.attn1 = nn.MultiheadAttention(hidden_dim, heads, batch_first=True, dropout=0.1)
        self.attn2 = nn.MultiheadAttention(hidden_dim, heads, batch_first=True, dropout=0.1)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim * 2, hidden_dim)
        )
        self.norm3 = nn.LayerNorm(hidden_dim)
        self.out = nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index):
        # x: [num_nodes, in_dim]
        # edge_index not used in this simple version; we only use node features
        # In a full TGAT, edge_index would be used for graph convolution.
        # Here we treat the graph as a set of nodes and use self-attention across all nodes.
        num_nodes = x.shape[0]
        t = torch.arange(num_nodes, dtype=torch.float32)
        t_enc = self.time_enc(t)  # [num_nodes, time_dim]
        h = torch.cat([x, t_enc], dim=-1)
        h = F.relu(self.in_proj(h))  # [num_nodes, hidden_dim]
        # Self-attention over all nodes (simulate graph context)
        h_seq = h.unsqueeze(0)  # [1, num_nodes, hidden_dim]
        h2, _ = self.attn1(h_seq, h_seq, h_seq)
        h = self.norm1(h + h2.squeeze(0))
        h2, _ = self.attn2(h_seq, h_seq, h_seq)
        h = self.norm2(h + h2.squeeze(0))
        h = self.norm3(h + self.ffn(h))
        return torch.sigmoid(self.out(h)).squeeze()  # [num_nodes]

# Instantiate model
in_dim = node_feats.shape[1]
model = TGAT(in_dim=in_dim, hidden_dim=128, heads=4, time_dim=64)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

11. Training Loop

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
criterion = torch.nn.BCELoss()

def run_epoch(indices, train=True):
    model.train() if train else model.eval()
    total_loss = 0
    correct1 = correct3 = total = 0
    with torch.set_grad_enabled(train):
        for idx in indices:
            x, edge_idx, events, label_idx = temporal_dataset[idx]
            # Update node features with alarm activity
            x_new = x.clone()
            for node, rel_time, sev in events:
                if node < x_new.shape[0]:
                    x_new[node, -2] = sev
                    x_new[node, -1] = 1.0 / (1 + rel_time / 60.0)  # recency
            out = model(x_new, edge_idx)
            target = torch.zeros_like(out)
            target[label_idx] = 1.0
            loss = criterion(out, target)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            total += 1
            # Metrics
            scores = out.detach().cpu().numpy()
            pred1 = np.argmax(scores)
            pred3 = np.argsort(scores)[::-1][:3]
            if pred1 == label_idx:
                correct1 += 1
            if label_idx in pred3:
                correct3 += 1
    return total_loss / max(total, 1), correct1 / max(total, 1), correct3 / max(total, 1)

# Training
EPOCHS = 30
best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': [], 'val_acc1': [], 'val_acc3': []}

print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Val Loss':>8} | {'Acc@1':>7} | {'Acc@3':>7}")
print("-" * 50)

for epoch in range(1, EPOCHS+1):
    tr_loss, _, _ = run_epoch(train_idx, train=True)
    val_loss, acc1, acc3 = run_epoch(val_idx, train=False)
    scheduler.step(val_loss)
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(val_loss)
    history['val_acc1'].append(acc1)
    history['val_acc3'].append(acc3)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'model/tgat_rca.pt')
    if epoch % 5 == 0 or epoch == 1:
        print(f"{epoch:>5} | {tr_loss:>10.4f} | {val_loss:>8.4f} | {acc1:>7.2%} | {acc3:>7.2%}")

print(f"\nBest val loss: {best_val_loss:.4f}")
model.load_state_dict(torch.load('model/tgat_rca.pt'))

12. Inference Engine

In [ ]:
def predict_rca(incident_alarms_df, model, graph_data, ne_encoder, severity_map, top_k=3):
    model.eval()
    with torch.no_grad():
        inc_alarms = incident_alarms_df.sort_values('OPEN_TIME')
        t0 = inc_alarms['OPEN_TIME'].iloc[0]
        events = []
        for _, row in inc_alarms.iterrows():
            ne_name = str(row.get('ENTITY_NAME', ''))
            if ne_name not in ne_encoder.classes_:
                continue
            node_idx = ne_encoder.transform([ne_name])[0]
            rel_time = (row['OPEN_TIME'] - t0).total_seconds()
            sev = severity_map.get(str(row.get('SEVERITY', 'WARNING')), 1)
            events.append((node_idx, rel_time, sev))
        if not events:
            return []
        x = graph_data.x.clone()
        for node, rel_time, sev in events:
            if node < x.shape[0]:
                x[node, -2] = sev
                x[node, -1] = 1.0 / (1 + rel_time / 60.0)
        out = model(x, graph_data.edge_index)
        scores = out.cpu().numpy()
        top_indices = np.argsort(scores)[::-1][:top_k]
        results = []
        for rank, idx in enumerate(top_indices):
            ne_name = ne_encoder.inverse_transform([idx])[0]
            results.append({
                'rank': rank+1,
                'ne_id': int(idx),
                'ne_name': ne_name,
                'confidence': float(round(scores[idx], 4))
            })
        return results

def build_rca_report(incident_id, predictions, alarm_df, ne_df):
    if not predictions:
        return {'incident_id': incident_id, 'error': 'No root cause found'}
    root = predictions[0]
    root_ne_name = root['ne_name']
    incident_alarms = alarm_df[alarm_df['INCIDENT_ID'] == incident_id]
    ne_info = ne_df[ne_df['NE_NAME'] == root_ne_name].iloc[0] if not ne_df.empty else None
    location = ne_info.get('GEOGRAPHY_L1_NAME', 'Unknown') if ne_info is not None else 'Unknown'
    return {
        'incident_id': incident_id,
        'root_cause_ne': root_ne_name,
        'confidence': root['confidence'],
        'top_candidates': predictions,
        'impacted_services': [],  # can be enriched from other tables
        'location': location,
        'alarm_count': len(incident_alarms),
        'alarm_types': incident_alarms['ALARM_NAME'].value_counts().head(3).to_dict()
    }

# Test on a sample incident
if not labels_df.empty:
    sample_inc = labels_df['INCIDENT_ID'].iloc[0]
    sample_alarms = alarm_clean[alarm_clean['INCIDENT_ID'] == sample_inc]
    preds = predict_rca(sample_alarms, model, graph_data, ne_encoder, severity_map)
    report = build_rca_report(sample_inc, preds, alarm_clean, ne_clean)
    print(json.dumps(report, indent=2))